In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

train = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/train.csv')
test  = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')
layout = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/layout_info.csv')

print(f'train: {train.shape}, test: {test.shape}, layout: {layout.shape}')

train: (250000, 94), test: (50000, 93), layout: (300, 15)


## 1. 전처리 - Scenario-level Aggregation

In [2]:
# Scenario-level 통계 (train 기준으로 생성 → test에 매핑)
agg_features = train.groupby('scenario_id').agg(
    sc_battery_mean_avg   = ('battery_mean', 'mean'),
    sc_low_battery_max    = ('low_battery_ratio', 'max'),
    sc_congestion_avg     = ('congestion_score', 'mean'),
    sc_order_inflow_avg   = ('order_inflow_15m', 'mean'),
    sc_order_inflow_max   = ('order_inflow_15m', 'max'),
    sc_robot_charging_avg = ('robot_charging', 'mean'),
).reset_index()

# Layout merge (building_age_years 제외 - 불필요 컬럼)
layout_clean = layout[['layout_id', 'layout_type', 'aisle_width_avg',
                        'intersection_count', 'one_way_ratio', 'pack_station_count',
                        'charger_count', 'layout_compactness', 'zone_dispersion',
                        'robot_total', 'floor_area_sqm', 'ceiling_height_m',
                        'fire_sprinkler_count', 'emergency_exit_count']]

# Train merge
train = train.merge(agg_features, on='scenario_id', how='left')
train = train.merge(layout_clean, on='layout_id', how='left')

# Test merge
test = test.merge(agg_features, on='scenario_id', how='left')
test = test.merge(layout_clean, on='layout_id', how='left')

print(f'train: {train.shape}, test: {test.shape}')
print('중복 컬럼:', [c for c in train.columns if '_x' in c or '_y' in c])

train: (250000, 113), test: (50000, 112)
중복 컬럼: []


## 2. 전처리 - 결측치 처리

In [3]:
TARGET = 'avg_delay_minutes_next_30m'

# 수치형 컬럼 분리
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_no_target = [c for c in numeric_cols if c != TARGET]
numeric_cols_test = test.select_dtypes(include=[np.number]).columns.tolist()

# 결측 indicator 생성 (결측 자체가 정보)
missing_cols = [col for col in numeric_cols_no_target if train[col].isnull().any()]
for col in missing_cols:
    train[f'{col}_missing'] = train[col].isnull().astype(int)
    test[f'{col}_missing']  = test[col].isnull().astype(int)

print(f'결측 indicator 추가: {len(missing_cols)}개')

# Median imputation (train fit → test transform, leakage 방지)
imputer = SimpleImputer(strategy='median')
train[numeric_cols_no_target] = imputer.fit_transform(train[numeric_cols_no_target])
test[numeric_cols_test]       = imputer.transform(test[numeric_cols_test])

print(f'결측치 잔여 - train: {train.isnull().sum().sum()}, test: {test.isnull().sum().sum()}')
print(f'train: {train.shape}, test: {test.shape}')

결측 indicator 추가: 86개
결측치 잔여 - train: 0, test: 0
train: (250000, 199), test: (50000, 198)


## 3. Feature Engineering

In [4]:
def add_features(df):
    new_feats = pd.DataFrame(index=df.index)
    
    # Battery-Robot 상호작용
    new_feats['battery_robot_stress']  = df['low_battery_ratio'] * df['robot_charging']
    new_feats['charge_pressure']       = df['charge_queue_length'] / (df['robot_idle'] + 1)
    new_feats['effective_robot_ratio'] = df['robot_active'] / (df['robot_active'] + df['robot_charging'] + 1)
    
    # 주문-용량 압박
    new_feats['order_robot_ratio'] = df['order_inflow_15m'] / (df['robot_active'] + 1)
    new_feats['order_complexity']  = df['order_inflow_15m'] * df['avg_items_per_order']
    new_feats['urgent_pressure']   = df['urgent_order_ratio'] * df['order_inflow_15m']
    
    # Layout 연계
    new_feats['robots_per_area']     = df['robot_total'] / df['floor_area_sqm']
    new_feats['charger_sufficiency'] = df['charger_count'] / df['robot_total']
    new_feats['pack_capacity_ratio'] = df['pack_utilization'] / (df['pack_station_count'] + 1)
    
    # Shift hour 주기성
    new_feats['shift_hour_sin']  = np.sin(2 * np.pi * df['shift_hour'] / 24)
    new_feats['shift_hour_cos']  = np.cos(2 * np.pi * df['shift_hour'] / 24)
    new_feats['late_shift_flag'] = (df['shift_hour'] >= 18).astype(int)
    
    return pd.concat([df, new_feats], axis=1).copy()

train = add_features(train)
test  = add_features(test)

print(f'train: {train.shape}, test: {test.shape}')

train: (250000, 211), test: (50000, 210)


## 4. 인코딩 & 불필요 컬럼 제거

In [5]:
# layout_type 인코딩
le = LabelEncoder()
train['layout_type'] = le.fit_transform(train['layout_type'])
test['layout_type']  = le.transform(test['layout_type'])

# ID, layout_id, scenario_id 제거
drop_cols = ['ID', 'layout_id', 'scenario_id']
train = train.drop(columns=drop_cols)
test  = test.drop(columns=drop_cols)

print(f'train: {train.shape}, test: {test.shape}')
print('object 컬럼 잔여:', train.select_dtypes(include='object').columns.tolist())

train: (250000, 208), test: (50000, 207)
object 컬럼 잔여: []


## 5. 모델 학습 optuna

In [7]:
mae_params = {
    'n_estimators'     : 10000,
    'learning_rate'    : 0.021624908800870646,
    'num_leaves'       : 452,
    'max_depth'        : 12,
    'min_child_samples': 39,
    'subsample'        : 0.8727893828993127,
    'colsample_bytree' : 0.696560164295319,
    'reg_alpha'        : 0.08039762424171229,
    'reg_lambda'       : 0.1561954924853681,
    'objective'        : 'mae',
    'metric'           : 'mae',
    'random_state'     : 42,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

X = train.drop(columns=[TARGET])
y = train[TARGET]  # log1p 제거, 원본 스케일 그대로

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_mae = lgb.LGBMRegressor(**mae_params)
model_mae.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)]
)

joblib.dump(model_mae, 'C:/Users/user/dacon/Smart_Logistics/models/lgbm_mae.pkl')

y_pred = model_mae.predict(X_val)
mae = np.mean(np.abs(y_val - y_pred))
print(f'Validation MAE: {mae:.4f}')

Training until validation scores don't improve for 100 rounds
[500]	valid_0's l1: 6.5008
[1000]	valid_0's l1: 6.0033
[1500]	valid_0's l1: 5.83087
[2000]	valid_0's l1: 5.74704
[2500]	valid_0's l1: 5.67192
[3000]	valid_0's l1: 5.6208
[3500]	valid_0's l1: 5.56903
[4000]	valid_0's l1: 5.54791
[4500]	valid_0's l1: 5.52956
[5000]	valid_0's l1: 5.51799
[5500]	valid_0's l1: 5.50158
[6000]	valid_0's l1: 5.48923
[6500]	valid_0's l1: 5.48022
[7000]	valid_0's l1: 5.4749
[7500]	valid_0's l1: 5.46448
[8000]	valid_0's l1: 5.45848
[8500]	valid_0's l1: 5.45579
[9000]	valid_0's l1: 5.45243
[9500]	valid_0's l1: 5.44956
[10000]	valid_0's l1: 5.44691
Did not meet early stopping. Best iteration is:
[9998]	valid_0's l1: 5.44691
Validation MAE: 5.4469


In [8]:
test_id = pd.read_csv('C:/Users/user/dacon/Smart_Logistics/data/test.csv')['ID']

y_test_pred = model_mae.predict(test)

submission = pd.DataFrame({
    'ID'                        : test_id,
    'avg_delay_minutes_next_30m': y_test_pred
})

submission.to_csv('C:/Users/user/dacon/Smart_Logistics/results/submission_mae.csv', index=False)
print(submission.head())

            ID  avg_delay_minutes_next_30m
0  TEST_000000                   11.590278
1  TEST_000001                   11.573433
2  TEST_000002                   11.965928
3  TEST_000003                   12.250261
4  TEST_000004                   12.914458
